In [7]:
import pandas as pd
import matplotlib.pyplot as plt

# Cargar los datos
df = pd.read_csv("C:\\Users\\eulal\\OneDrive\\Escritorio\\laloseasontwo\\Minería de datos\\Data Cleaning\\netflix_titles.csv")
# Estadísticas descriptivas básicas
print("Información del dataset:")
print(df.info())

print("\nEstadísticas descriptivas para columnas numéricas:")
print(df.describe())

print("\nValores únicos por columna categórica:")
for col in df.select_dtypes(include=['object']).columns:
    print(f"{col}: {df[col].nunique()} valores únicos")

Información del dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB
None

Estadísticas descriptivas para columnas numéricas:
       release_year
count   8807.000000
mean    2014.180198
std        8.819312
min     1925.000000
25%     2013.000000
50%     2017.000000
75%     2019.000000
max     20

Asi es como dejare mis entidades para este modelo de datos:

CONTENIDO(
  show_id (PK),
  title,
  type,
  date_added,
  release_year,
  rating,
  description
)


PERSONA(
  persona_id (PK),
  nombre,
  tipo (director/actor)
)

PAÍS(
  pais_id (PK),
  nombre
)

GÉNERO(
  genero_id (PK),
  nombre
)

Asi es como dejare mis relaciones para este modelo de datos:


DIRIGE(
  show_id (FK),
  persona_id (FK)
)


ACTÚA(
  show_id (FK),
  persona_id (FK)
)



PRODUCIDO_EN(
  show_id (FK),
  pais_id (FK)
)


CLASIFICADO_COMO(
  show_id (FK),
  genero_id (FK)
)

In [ ]:
import pandas as pd
from itertools import chain

# 1. Tabla CONTENIDO
contenido = df[['show_id', 'title', 'type', 'date_added', 'release_year', 'rating', 'description']]

# 2. Tabla PERSONA (directores y actores)
# Para directores
directores = df['director'].str.split(', ', expand=True).stack().reset_index(drop=True).to_frame('nombre')
directores['tipo'] = 'director'

# Para actores
actores = df['cast'].str.split(', ', expand=True).stack().reset_index(drop=True).to_frame('nombre')
actores['tipo'] = 'actor'

# Combinar
personas = pd.concat([directores, actores]).drop_duplicates().reset_index(drop=True)
personas['persona_id'] = personas.index + 1

# 3. Tabla PAÍS
paises = df['country'].str.split(', ', expand=True).stack().reset_index(drop=True).to_frame('nombre')
paises = paises.drop_duplicates().reset_index(drop=True)
paises['pais_id'] = paises.index + 1

# 4. Tabla GÉNERO
generos = df['listed_in'].str.split(', ', expand=True).stack().reset_index(drop=True).to_frame('nombre')
generos = generos.drop_duplicates().reset_index(drop=True)
generos['genero_id'] = generos.index + 1

In [13]:
print(generos)

                          nombre  genero_id
0                  Documentaries          1
1         International TV Shows          2
2                      TV Dramas          3
3                   TV Mysteries          4
4                 Crime TV Shows          5
5          TV Action & Adventure          6
6                     Docuseries          7
7                     Reality TV          8
8              Romantic TV Shows          9
9                    TV Comedies         10
10                     TV Horror         11
11      Children & Family Movies         12
12                        Dramas         13
13            Independent Movies         14
14          International Movies         15
15              British TV Shows         16
16                      Comedies         17
17     Spanish-Language TV Shows         18
18                     Thrillers         19
19               Romantic Movies         20
20              Music & Musicals         21
21                 Horror Movies

In [17]:
# Tabla DIRIGE (relación entre contenido y directores)
dirige = (
    df['director'].str.split(', ', expand=True)
    .stack()
    .reset_index()
    .rename(columns={'level_0': 'show_idx', 0: 'nombre'})
)
dirige = pd.merge(
    dirige,
    personas[personas['tipo'] == 'director'],
    on='nombre',
    how='left'
)[['show_idx', 'persona_id']]
dirige['show_id'] = df.loc[dirige['show_idx'], 'show_id'].values

# Tabla ACTUA (relación entre contenido y actores)
actua = (
    df['cast'].str.split(', ', expand=True)
    .stack()
    .reset_index()
    .rename(columns={'level_0': 'show_idx', 0: 'nombre'})
)
actua = pd.merge(
    actua,
    personas[personas['tipo'] == 'actor'],
    on='nombre',
    how='left'
)[['show_idx', 'persona_id']]
actua['show_id'] = df.loc[actua['show_idx'], 'show_id'].values

# Tabla PRODUCIDO_EN (relación entre contenido y países)
producido = (
    df['country'].str.split(', ', expand=True)
    .stack()
    .reset_index()
    .rename(columns={'level_0': 'show_idx', 0: 'nombre'})
)
producido = pd.merge(
    producido,
    paises,
    on='nombre',
    how='left'
)[['show_idx', 'pais_id']]
producido['show_id'] = df.loc[producido['show_idx'], 'show_id'].values

# Tabla CLASIFICADO (relación entre contenido y géneros)
clasificado = (
    df['listed_in'].str.split(', ', expand=True)
    .stack()
    .reset_index()
    .rename(columns={'level_0': 'show_idx', 0: 'nombre'})
)
clasificado = pd.merge(
    clasificado,
    generos,
    on='nombre',
    how='left'
)[['show_idx', 'genero_id']]
clasificado['show_id'] = df.loc[clasificado['show_idx'], 'show_id'].values

In [19]:
def get_directores(titulo):
    try:
        # Verificar si el título existe
        if titulo not in contenido['title'].values:
            print(f"Error: El título '{titulo}' no existe en el dataset")
            print("Algunos títulos disponibles:", contenido['title'].head(5).tolist())
            return None
        
        # Obtener show_id del título
        show_id = contenido[contenido['title'] == titulo]['show_id'].iloc[0]
        
        # Obtener IDs de directores
        dir_ids = dirige[dirige['show_id'] == show_id]['persona_id']
        
        # Verificar si hay directores
        if dir_ids.empty:
            print(f"El título '{titulo}' no tiene directores registrados")
            return None
            
        # Obtener información completa de los directores
        directores = personas[personas['persona_id'].isin(dir_ids)]
        
        return directores[['nombre', 'tipo']]
    
    except Exception as e:
        print(f"Error inesperado: {str(e)}")
        return None

In [20]:
print("Algunos títulos en el dataset:")
print(contenido['title'].head(10).tolist())

Algunos títulos en el dataset:
['Dick Johnson Is Dead', 'Blood & Water', 'Ganglands', 'Jailbirds New Orleans', 'Kota Factory', 'Midnight Mass', 'My Little Pony: A New Generation', 'Sankofa', 'The Great British Baking Show', 'The Starling']


In [21]:
# Ejemplo con un título que sí existe (usa uno de tu dataset)
directores = get_directores('Ganglands')  # Cambia por un título de tu lista
if directores is not None:
    print("\nDirectores encontrados:")
    print(directores)


Directores encontrados:
            nombre      tipo
1  Julien Leclercq  director
